# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Purpose of the Validation Stage
This stage serves as a strict quality gate. We will validate whether the `relabeled_cyberbullying_dataset.csv` from the previous stage meets all criteria required for the upcoming text preprocessing stage (`05_preprocessing.ipynb`).

## Core Principle
**Validation is NOT Relabeling.** 
This notebook will only **inspect and report** on data quality. It will **not** silently fix, delete, or modify any data to force a passing result. If the data fails critical checks, the issue must be resolved in earlier pipeline stages.


# 2. VALIDATION OBJECTIVES

This stage aims to verify:
- Dataset existence and readability.
- Schema correctness and required columns presence.
- Data types matching expectations.
- Acceptable levels of missing values.
- Text integrity (no empty or whitespace-only strings).
- Duplicate conditions (exact, text-level, and conflicting labels).
- Label validity (strict adherence to official taxonomy).
- Label consistency.
- Provenance preservation (tracking origin datasets).
- Overall readiness for preprocessing.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_RELABELED_DIR = PROJECT_ROOT / "data" / "relabeled"
DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
REPORTS_DIR = PROJECT_ROOT / "reports"

RELABELED_DATA_PATH = DATA_RELABELED_DIR / "relabeled_cyberbullying_dataset.csv"
VALIDATED_DATA_PATH = DATA_VALIDATED_DIR / "validated_cyberbullying_dataset.csv"
VALIDATION_SUMMARY_PATH = REPORTS_DIR / "validation_summary.csv"

# Target Columns based on docs
TEXT_COL = "text"
LABEL_COL = "cyberbullying_type"
SEVERITY_COL = "severity_level"
PROVENANCE_COLS = ["source_dataset", "original_row_id"]
OFFICIAL_LABELS = ["normal", "insult", "harassment", "threat", "hate_speech"]

DATA_VALIDATED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Relabeled Data Path: {RELABELED_DATA_PATH}")


Relabeled Data Path: /home/zapp/Kampus/PM-NEW/data/relabeled/relabeled_cyberbullying_dataset.csv


In [3]:
# 5. LOAD RELABELED DATASET

if not RELABELED_DATA_PATH.exists():
    raise FileNotFoundError(f"FAIL: Relabeled dataset not found at {RELABELED_DATA_PATH}. Please run 03_relabeling.ipynb first.")

df = pd.read_csv(RELABELED_DATA_PATH)
print(f"Dataset successfully loaded.")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
display(df.head())


Dataset successfully loaded.
Shape: 42954 rows, 8 columns
Columns: ['text', 'cyberbullying_type', 'severity_level', 'original_cyberbullying_type', 'original_severity_level', 'source_dataset', 'source_file', 'original_row_id']


,text,cyberbullying_type,severity_level,original_cyberbullying_type,original_severity_level,source_dataset,source_file,original_row_id
0,setiap orang adalah seorang gadis yang akan me...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,0
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1
2,karena beberapa orang tidak ada yang lebih bai...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,2
3,bro aku pelatih jv tahun lalu di skyline dan a...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,3
4,wanitawanita ini benarbenar mengingatkan saya ...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,4


In [4]:
# 6. VALIDATION RESULT FORMAT

# We use a structured list to collect validation results
validation_results = []

def add_check(check_id, check_name, category, status, severity, affected_count, details, recommendation):
    validation_results.append({
        "check_id": check_id,
        "check_name": check_name,
        "category": category,
        "status": status,
        "severity": severity,
        "affected_count": affected_count,
        "details": details,
        "recommendation": recommendation
    })

print("Validation reporting structure initialized.")


Validation reporting structure initialized.


In [5]:
# 7. SCHEMA VALIDATION

is_empty = df.empty
row_count = len(df)

if is_empty or row_count == 0:
    add_check("V_SCH_01", "Dataset Not Empty", "Schema", "FAIL", "CRITICAL", 0, "Dataset contains 0 rows.", "Ensure previous stage outputted data.")
else:
    add_check("V_SCH_01", "Dataset Not Empty", "Schema", "PASS", "INFO", 0, f"Dataset contains {row_count} rows.", "None")

print(f"Schema Validation: {'FAIL' if is_empty else 'PASS'} ({row_count} rows)")


Schema Validation: PASS (42954 rows)


In [6]:
# 8. REQUIRED COLUMN VALIDATION

expected_columns = [TEXT_COL, LABEL_COL] + PROVENANCE_COLS
missing_cols = [col for col in expected_columns if col not in df.columns]

if missing_cols:
    add_check("V_COL_01", "Required Columns Exist", "Columns", "FAIL", "CRITICAL", len(missing_cols), f"Missing: {', '.join(missing_cols)}", "Fix pipeline to ensure columns exist.")
else:
    add_check("V_COL_01", "Required Columns Exist", "Columns", "PASS", "INFO", 0, "All required columns are present.", "None")

print(f"Required columns check: {'FAIL' if missing_cols else 'PASS'}")


Required columns check: PASS


In [7]:
# 9. DATA TYPE VALIDATION

type_issues = 0
if TEXT_COL in df.columns and not pd.api.types.is_string_dtype(df[TEXT_COL]):
    # some mixed types might show as object, we check if strictly non-string exists in Text Integrity
    pass

if LABEL_COL in df.columns and not pd.api.types.is_string_dtype(df[LABEL_COL]):
    type_issues += 1
    add_check("V_TYP_01", "Label Data Type", "Data Types", "FAIL", "CRITICAL", 0, f"Column {LABEL_COL} is not string-like.", "Convert label column to string.")
else:
    add_check("V_TYP_01", "Label Data Type", "Data Types", "PASS", "INFO", 0, f"{LABEL_COL} type is valid.", "None")

print(f"Type validation issues: {type_issues}")


Type validation issues: 0


In [8]:
# 10. MISSING VALUE VALIDATION

missing_text = df[TEXT_COL].isnull().sum() if TEXT_COL in df.columns else 0
missing_labels = df[LABEL_COL].isnull().sum() if LABEL_COL in df.columns else 0

if missing_text > 0:
    add_check("V_NA_01", "Missing Text", "Missing Values", "FAIL", "CRITICAL", missing_text, f"{missing_text} rows have null text.", "Remove or fix null texts in previous steps.")
else:
    add_check("V_NA_01", "Missing Text", "Missing Values", "PASS", "INFO", 0, "No null texts.", "None")

if missing_labels > 0:
    add_check("V_NA_02", "Missing Labels", "Missing Values", "FAIL", "CRITICAL", missing_labels, f"{missing_labels} rows have null labels.", "Resolve unlabeled data in relabeling stage.")
else:
    add_check("V_NA_02", "Missing Labels", "Missing Values", "PASS", "INFO", 0, "No null labels.", "None")

print(f"Missing Values - Text: {missing_text}, Labels: {missing_labels}")


Missing Values - Text: 0, Labels: 0


In [9]:
# 11. TEXT INTEGRITY VALIDATION

empty_str_count = 0
whitespace_count = 0

if TEXT_COL in df.columns:
    texts = df[TEXT_COL].dropna().astype(str)
    empty_str_count = (texts == "").sum()
    whitespace_count = texts.str.isspace().sum()

if empty_str_count > 0 or whitespace_count > 0:
    add_check("V_TXT_01", "Text Integrity", "Text", "FAIL", "CRITICAL", empty_str_count + whitespace_count, 
              f"{empty_str_count} empty strings, {whitespace_count} whitespace-only.", "Remove empty texts.")
else:
    add_check("V_TXT_01", "Text Integrity", "Text", "PASS", "INFO", 0, "Text integrity checks passed.", "None")

print(f"Text Integrity - Empty Strings: {empty_str_count}, Whitespace-only: {whitespace_count}")


Text Integrity - Empty Strings: 0, Whitespace-only: 0


In [10]:
# 12. DUPLICATE VALIDATION

exact_dupes = df.duplicated().sum()
text_dupes = df.duplicated(subset=[TEXT_COL]).sum() if TEXT_COL in df.columns else 0

conflicting_dupes = 0
if TEXT_COL in df.columns and LABEL_COL in df.columns:
    # Texts that appear multiple times with DIFFERENT labels
    grouped = df.groupby(TEXT_COL)[LABEL_COL].nunique()
    conflicting_texts = grouped[grouped > 1].index
    conflicting_dupes = df[df[TEXT_COL].isin(conflicting_texts)].shape[0]

if conflicting_dupes > 0:
    add_check("V_DUP_01", "Conflicting Label Duplicates", "Duplicates", "FAIL", "CRITICAL", conflicting_dupes,
              "Same text assigned multiple different labels.", "Resolve conflicts in relabeling queue.")
else:
    add_check("V_DUP_01", "Conflicting Label Duplicates", "Duplicates", "PASS", "INFO", 0, "No conflicting labels.", "None")

if exact_dupes > 0:
    add_check("V_DUP_02", "Exact Row Duplicates", "Duplicates", "WARNING", "WARNING", exact_dupes,
              f"{exact_dupes} exact duplicate rows found.", "Consider dropping exact duplicates during preprocessing.")
else:
    add_check("V_DUP_02", "Exact Row Duplicates", "Duplicates", "PASS", "INFO", 0, "No exact duplicates.", "None")

print(f"Duplicates - Exact: {exact_dupes}, Conflicting: {conflicting_dupes}")


Duplicates - Exact: 0, Conflicting: 0


In [11]:
# 13. LABEL VALIDATION

invalid_labels = 0
if LABEL_COL in df.columns:
    labels = df[LABEL_COL].dropna().astype(str)
    invalid_mask = ~labels.isin(OFFICIAL_LABELS)
    invalid_labels = invalid_mask.sum()
    
    if invalid_labels > 0:
        bad_examples = labels[invalid_mask].unique()
        add_check("V_LBL_01", "Taxonomy Compliance", "Labels", "FAIL", "CRITICAL", invalid_labels,
                  f"Found labels outside taxonomy: {bad_examples}", "Map these labels in relabeling stage.")
    else:
        add_check("V_LBL_01", "Taxonomy Compliance", "Labels", "PASS", "INFO", 0, "All labels comply with official taxonomy.", "None")
        
print(f"Invalid Labels: {invalid_labels}")


Invalid Labels: 0


In [12]:
# 14. LABEL DISTRIBUTION VALIDATION

if LABEL_COL in df.columns:
    dist = df[LABEL_COL].value_counts()
    missing_classes = [c for c in OFFICIAL_LABELS if c not in dist.index]
    
    print("Label Distribution:")
    print(dist)
    
    if missing_classes:
        add_check("V_LBL_02", "Missing Target Classes", "Labels", "WARNING", "WARNING", len(missing_classes),
                  f"Classes missing from data: {missing_classes}", "Note for model evaluation limitations.")
    else:
        add_check("V_LBL_02", "Missing Target Classes", "Labels", "PASS", "INFO", 0, "All official classes are represented.", "None")


Label Distribution:
cyberbullying_type
normal         28914
hate_speech     7147
harassment      5147
insult          1746
Name: count, dtype: int64


In [13]:
# 15. RELABELING AUDIT VALIDATION

audit_path = REPORTS_DIR / "relabeling_audit.csv"
if not audit_path.exists():
    add_check("V_AUD_01", "Audit Trail Exists", "Provenance", "WARNING", "WARNING", 0, "relabeling_audit.csv not found.", "Ensure relabeling notebook generated audit.")
else:
    add_check("V_AUD_01", "Audit Trail Exists", "Provenance", "PASS", "INFO", 0, "Audit trail found.", "None")
print("Audit check completed.")


Audit check completed.


In [14]:
# 16. DATA PROVENANCE VALIDATION

missing_prov = 0
for col in PROVENANCE_COLS:
    if col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            missing_prov += null_count
            
if missing_prov > 0:
    add_check("V_PRV_01", "Provenance Completeness", "Provenance", "WARNING", "WARNING", missing_prov,
              f"{missing_prov} rows have missing provenance data.", "Investigate data merge stage.")
else:
    add_check("V_PRV_01", "Provenance Completeness", "Provenance", "PASS", "INFO", 0, "Provenance data is complete.", "None")
print(f"Provenance missing values: {missing_prov}")


Provenance missing values: 0


In [15]:
# 17. VALIDATION RULE SUMMARY

val_df = pd.DataFrame(validation_results)
display(val_df)


,check_id,check_name,category,status,severity,affected_count,details,recommendation
0,V_SCH_01,Dataset Not Empty,Schema,PASS,INFO,0,Dataset contains 42954 rows.,None
1,V_COL_01,Required Columns Exist,Columns,PASS,INFO,0,All required columns are present.,None
2,V_TYP_01,Label Data Type,Data Types,PASS,INFO,0,cyberbullying_type type is valid.,None
3,V_NA_01,Missing Text,Missing Values,PASS,INFO,0,No null texts.,None
4,V_NA_02,Missing Labels,Missing Values,PASS,INFO,0,No null labels.,None
5,V_TXT_01,Text Integrity,Text,PASS,INFO,0,Text integrity checks passed.,None
6,V_DUP_01,Conflicting Label Duplicates,Duplicates,PASS,INFO,0,No conflicting labels.,None
7,V_DUP_02,Exact Row Duplicates,Duplicates,PASS,INFO,0,No exact duplicates.,None
8,V_LBL_01,Taxonomy Compliance,Labels,PASS,INFO,0,All labels comply with official taxonomy.,None
9,V_LBL_02,Missing Target Classes,Labels,WARNING,WARNING,1,Classes missing from data: ['threat'],Note for model evaluation limitations.


In [16]:
# 18. VALIDATION REPORT

pass_count = (val_df['status'] == 'PASS').sum()
warn_count = (val_df['status'] == 'WARNING').sum()
fail_count = (val_df['status'] == 'FAIL').sum()

print("="*50)
print("             VALIDATION REPORT")
print("="*50)
print(f"Dataset: {RELABELED_DATA_PATH}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print("-" * 50)
print(f"PASS:    {pass_count}")
print(f"WARNING: {warn_count}")
print(f"FAIL:    {fail_count}")
print("-" * 50)

final_status = "READY"
if fail_count > 0:
    final_status = "NOT READY"
elif warn_count > 0:
    final_status = "READY WITH WARNINGS"

print(f"OVERALL STATUS: {final_status}")
print("="*50)


             VALIDATION REPORT
Dataset: /home/zapp/Kampus/PM-NEW/data/relabeled/relabeled_cyberbullying_dataset.csv
Rows: 42954 | Columns: 8
--------------------------------------------------
PASS:    11
FAIL:    0
--------------------------------------------------
OVERALL STATUS: READY WITH WARNINGS


In [17]:
# 19. SAVE VALIDATED DATASET

if final_status in ["READY", "READY WITH WARNINGS"]:
    df.to_csv(VALIDATED_DATA_PATH, index=False)
    print(f"SUCCESS: Validated dataset saved to {VALIDATED_DATA_PATH}")
else:
    print(f"CRITICAL: Dataset is NOT READY. Fix the {fail_count} failing checks in the preceding notebooks.")
    print("No output file generated.")


SUCCESS: Validated dataset saved to /home/zapp/Kampus/PM-NEW/data/validated/validated_cyberbullying_dataset.csv


In [18]:
# 20. SAVE VALIDATION ARTIFACTS

val_df.to_csv(VALIDATION_SUMMARY_PATH, index=False)
print(f"Validation summary artifact saved to {VALIDATION_SUMMARY_PATH}")


Validation summary artifact saved to /home/zapp/Kampus/PM-NEW/reports/validation_summary.csv


# 21. FINAL VALIDATION DECISION

The dataset is currently evaluated as: **SEE FINAL STATUS IN CELL 18**.

- If **READY**: The dataset is perfectly aligned with requirements.
- If **READY WITH WARNINGS**: The dataset has minor issues (like exact duplicate rows) that should be handled in `05_preprocessing.ipynb`.
- If **NOT READY**: You **must** return to `03_relabeling.ipynb` (or earlier) to resolve the failing criteria (such as unresolved labels, conflicting duplicates, or missing values). Do not proceed to preprocessing until the dataset passes validation.


# How to Run This Notebook

1. Activate the project's virtual environment.
2. Ensure the relabeled dataset exists in: `data/relabeled/relabeled_cyberbullying_dataset.csv`
3. Ensure the relabeling stage has been completed: `notebooks/03_relabeling.ipynb`
4. Open: `notebooks/04_validation.ipynb`
5. Verify the configured input dataset path in Step 4.
6. Run the notebook from the first cell to the last cell.
7. Review the validation summary in Step 17.
8. Review failed and warning checks.
9. Review the generated validation report in Step 18.
10. Confirm whether the dataset is:
    - **READY**
    - **READY WITH WARNINGS**
    - **NOT READY**
11. If the dataset is ready, proceed to: `notebooks/05_preprocessing.ipynb`
